<a href="https://colab.research.google.com/github/Pratyay-B1swas/saas-revenue-churn-intelligence/blob/main/microsaas_product_and_revenue_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

customers = pd.read_csv('customers.csv')
revenue = pd.read_csv('revenue.csv')
subscriptions = pd.read_csv('subscriptions.csv')
activity = pd.read_csv('user_activity.csv')

print ("Dataset load done.")

Dataset load done.


In [2]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
customers['churn_date'] = pd.to_datetime(customers['churn_date'])
activity['activity_date'] = pd.to_datetime(activity['activity_date'])
customers['is_churned'] = customers['churn_date'].notnull().astype(int)

print("dates converted.")

dates converted.


In [3]:
user_activity_features = activity.groupby('customer_id').agg(total_logins = ('login_count', 'sum'), avg_logins = ('login_count', 'mean'), avg_features_used = ('features_used_count', 'mean'), total_support_ticktes = ('support_tickets_opened', 'sum')).reset_index()
df_master = pd.merge(customers, user_activity_features, on = 'customer_id', how = 'left')

print(df_master)
print("user angageme")

     customer_id signup_date   plan_type  monthly_fee  acquisition_cost  \
0           1001  2024-11-07       Basic           50                30   
1           1002  2024-06-06       Basic           50                30   
2           1003  2024-12-31       Basic           50                30   
3           1004  2024-11-21         Pro          200               100   
4           1005  2024-08-16         Pro          200               100   
..           ...         ...         ...          ...               ...   
995         1996  2025-02-18         Pro          200               100   
996         1997  2025-02-12         Pro          200               100   
997         1998  2024-01-13       Basic           50                30   
998         1999  2025-04-08  Enterprise          500               200   
999         2000  2024-07-24         Pro          200               100   

    churn_date  is_churned  total_logins  avg_logins  avg_features_used  \
0          NaT          

In [6]:
features = ['monthly_fee', 'acquisition_cost', 'avg_logins', 'avg_features_used', 'total_support_ticktes']
x = df_master[features]
y = df_master['is_churned']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)
clf = RandomForestClassifier(n_estimators = 100, random_state = 42)
clf.fit(x_train, y_train)

print("model trained")

df_master['churn_risk_score'] = (clf.predict_proba(x)[:, 1] *100).round(2)



model trained


In [7]:
df_master['churn_risk_score'] = (clf.predict_proba(x)[:, 1] *100).round(2)
df_master['at_risk_mrr_loss'] = ((df_master['churn_risk_score'] / 100.0)* df_master['monthly_fee']).round(2)
def recommend_action(row):
  risk = row['churn_risk_score']
  mrr  = row['monthly_fee']
  tickets = row['total_support_ticktes']
  if risk >= 70:
    if mrr >= 200:
      return "High priority: 1-on-1 call and 25% Renewal Discount."
    else:
      return "High Risk utomated Re-engagement Email and 15% discount"
  elif risk >=40:
    if tickets >= 3:
      return "Medium Risk: Assignsupport specialist for ticket resulation."
    else:
      return "Medimu Risk: Send fearutes tuturial and product tip campaign"
  else:
    return "Low Risk: Target for pro/enterprise plan upsell"

df_master['recommended_action'] = df_master.apply(recommend_action, axis = 1)



In [12]:
total_mrr = df_master[df_master['is_churned'] ==0]['monthly_fee'].sum()
mrr_at_risk = df_master[df_master['churn_risk_score']>= 60]['monthly_fee'].sum()
avg_cac = df_master['acquisition_cost'].mean()
overall_churn_rate = (df_master['is_churned'].sum() / len(df_master)) * 100

print(f"Total active MRR: ${total_mrr:,.2f}")
print(f"High-Risk MRR at threat: ${mrr_at_risk:,.2f}")
print(f"Average customer acquisition cost: ${avg_cac:,.2f}")
print(f"Overall customer churn rate: {overall_churn_rate:.2f}%")



Total active MRR: $207,950.00
High-Risk MRR at threat: $37,500.00
Average customer acquisition cost: $110.11
Overall customer churn rate: 16.80%


In [13]:
df_master.to_csv('final_assa_product_analysis.csv', index = False)

In [ ]:
import streamlit as st
import plotly.express as px

st.set_page_config(page_title = "Micro-SaaS Intelligence Hub", layout = "wide")

@st.cache_data
def load_data():
  return pd.read_csv('final_assa_product_analysis.csv')

df = load_data()
st.sidebar.header("Filter analysis")
selected_plans = st.sidebar.multiselect("Select subscription plan: ", options = df['plan_type'].unique(), default = df['plan_type'].unique())

filtered_df = df[df['plan_type'].isin(selected_plans)]

print("Sidebar filters showed")

st.title("Micro-Saas Revenue & Churn Intelligence Hub")
st.markdown("Automation AT-Rist Revenur Deetection & Action Engine Dashboard")
col1, col2, col3, col4 = st.columns(4)
active_mrr = filtered_df[filtered_df['is_churned'] == 0]['monthly_fee'].sum()
risk_mrr = filtered_df[filtered_df['churn_risk_score'] >= 60]['monthly_fee'].sum()
avg_cac = filtered_df['acquisition_cost'].mean()
churn_rate = (filtered_df['is_churned'].sum() / len(filtered_df)) * 100

col1.metric("Total active MRR", f"${active_mrr:,.2f}")
col2.metric("MRR at risk", f"${risk_mrr:,.2f}", delta = "High Threat", delta_color = "inverse")
col3.metric("Avg acquisition cost", f"${avg_cac:.2f}")
col4.metric("Overall churn rate", f"{churn_rate:.2f}%")
st.divider()

print("Dashboard header complete.")

c1, c2 = st.columns(2)
with c1:
  fig_risk = px.histogram(
      filtered_df,x = "churn_risk_score",
      color = "plan_type",
      title = "Churn Risk Score Distribution by Plan",
      barmode = "overlay"
  )
  st.plotly_chart(fig_risk, use_container_width=True)

with c2:
  fig_scatter = px.scatter(
      filtered_df,
      x = "avg_logins",y = "churn_risk_score",
      color = "plan_type",
      size = "monthly_fee",
      title = "User Engagement vs Churn Risk Score"
  )
  st.plotly_chart(fig_scatter, use_container_width=True)

  print("Visually represented")

  st.subheader("High-Risk Customers & Automated Recommendations")
  high_risk_df = filtered_df[filtered_df['churn_risk_score'] >= 60][['customer_id', 'plan_type', 'monthly_fee', 'churn_risk_score', 'at_risk_mrr_loss', 'recommended_action']]

  st.dataframe(
      high_risk_df.sort_values(by = 'churn_risk_score', ascending=False), use_container_width= True
  )


In [ ]:
%matplotlib inline

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
sns.histplot(
    data=df,
    x="churn_risk_score",
    hue="plan_type",
    kde=True,
    ax=axes[0, 0],
    palette="Set2",
)
axes[0, 0].set_title(
    "Churn Risk Score Distribution by Plan", fontsize=12, fontweight="bold"
)
axes[0, 0].set_xlabel("Churn Risk Score")
axes[0, 0].set_ylabel("Customer Count")
sns.scatterplot(
    data=df,
    x="avg_logins",
    y="churn_risk_score",
    hue="plan_type",
    size="monthly_fee",
    sizes=(20, 200),
    ax=axes[0, 1],
    palette="Set2",
)
axes[0, 1].set_title(
    "Engagement (Avg Logins) vs Churn Risk Score",
    fontsize=12,
    fontweight="bold",
)
axes[0, 1].set_xlabel("Average Weekly Logins")
axes[0, 1].set_ylabel("Churn Risk Score (%)")
sns.boxplot(
    data=df, x="plan_type", y="at_risk_mrr_loss", ax=axes[1, 0], palette="Set2"
)
axes[1, 0].set_title(
    "Potential MRR Loss per Customer by Plan", fontsize=12, fontweight="bold"
)
axes[1, 0].set_xlabel("Plan Type")
axes[1, 0].set_ylabel("At-Risk MRR Loss ($)")
action_counts = df["recommended_action"].value_counts()
axes[1, 1].pie(
    action_counts,
    labels=action_counts.index,
    autopct="%1.1f%%",
    colors=sns.color_palette("Set2"),
    startangle=140,
)
axes[1, 1].set_title(
    "Automated Recommendation Engine Breakdown",
    fontsize=12,
    fontweight="bold",
)

plt.tight_layout()
plt.savefig("dashboard_preview.png", dpi=300)
plt.show()